# Cell-Akamai · concurrency evidence replay

**Human/machine-run Jupyter companion.** This notebook verifies the frozen ECA input hash and reads locally committed concurrency summaries. It does not run models, change an Akamai deployment, score the oracle or claim training. Without run receipts it displays `NOT_TESTED`. Results are a nonconfirmatory pilot until independently scored.

## 1. Dataset contract / source identity

In [ ]:
from pathlib import Path
import json, hashlib, sys
ROOT=Path.cwd()
if not (ROOT / "dataset/eca_pilot_manifest_v1.json").exists(): ROOT=ROOT.parent
assert (ROOT / "dataset/eca_pilot_manifest_v1.json").exists(), "Start from repository root/notebooks"
manifest=json.loads((ROOT / "dataset/eca_pilot_manifest_v1.json").read_text())
b=(ROOT/manifest["path"]).read_bytes()
assert "sha256:"+hashlib.sha256(b).hexdigest()==manifest["sha256"]
items=[json.loads(line) for line in b.splitlines() if line]
assert len(items)==32 and len(set(r["item_id"] for r in items))==32
print("DATASET_SHA_VERIFIED", manifest["sha256"], "ITEMS",len(items), "MODE", manifest["status"])

## 2. Load append-only concurrent summary receipts (no model requests)

In [ ]:
import importlib.util
spec=importlib.util.spec_from_file_location("dash", ROOT/"tools/concurrency/serve_dashboard.py")
dash=importlib.util.module_from_spec(spec); spec.loader.exec_module(dash)
import os
RUNS=Path(os.environ.get("CELL_AKAMAI_RUNS", ROOT/"runs/concurrency"))
raw=dash.load_sanitized(RUNS)
rows=raw["rows"]
print("VALIDATED SUMMARY ROWS AVAILABLE",len(rows))
if not rows: print("NOT_TESTED — no real concurrency model receipts in this notebook environment")
else:
 for r in rows[:12]:
  print(r["backend"],r["stage"],r["arm"],"c=",r["concurrency"],"tok/s=",r["aggregate_completion_tokens_per_s"],"p95ms=",r["latency_p95_ms"])

## 3. Throughput versus concurrency

Aggregate completion tokens divided by the cell-level wall clock duration, **not** stream chunk count. Streamed inter-request tokens/s² cannot be reliably inferred for overlapping requests.

In [ ]:
if rows:
 import matplotlib.pyplot as plt
 fig,ax=plt.subplots(figsize=(9,5))
 groups=sorted(set((r["backend"],r["stage"],r["arm"]) for r in rows))
 for group in groups:
  rr=sorted((r for r in rows if (r["backend"],r["stage"],r["arm"])==group and isinstance(r["aggregate_completion_tokens_per_s"],(int,float))),key=lambda x:x["concurrency"])
  if rr:ax.plot([r["concurrency"] for r in rr],[r["aggregate_completion_tokens_per_s"] for r in rr],marker="o",label="/".join(group))
 ax.set(xlabel="Concurrent ECA items",ylabel="Aggregate completion tokens/s",title="Measured pilot throughput (backend/config/arm separate)")
 ax.legend(fontsize=7);ax.grid(alpha=0.3);plt.show()
else: print("NOT_TESTED — throughput chart intentionally omitted")

## 4. Latency tradeoffs and scientific claim ceiling

In [ ]:
if rows:
 import matplotlib.pyplot as plt
 fig,ax=plt.subplots(figsize=(9,5))
 for group in groups:
  rr=sorted((r for r in rows if (r["backend"],r["stage"],r["arm"])==group and isinstance(r["latency_p95_ms"],(int,float))),key=lambda x:x["concurrency"])
  if rr:ax.plot([r["concurrency"] for r in rr],[r["latency_p95_ms"] for r in rr],marker="o",label="/".join(group))
 ax.set(xlabel="Concurrent ECA items",ylabel="Per-item latency p95 (ms)",title="Pilot latency versus concurrency")
 ax.legend(fontsize=7);ax.grid(alpha=0.3);plt.show()
else: print("NOT_TESTED — latency chart intentionally omitted")
print("ANTICUBE NOT_COMPUTED | ΔG* NOT_COMPUTED | SIGNATURE NOT_SIGNED | MMR NOT_COMMITTED")

## 5. Lane B science (independently scored after answer commitment)

Rows appear only for panels whose `answers.jsonl`, config and frozen-dataset digests verified and whose 4-arm × item coverage is complete; refused panels stay listed. Akamai vLLM configurations and local Ollama models are reported separately. B1 is `GENESIS_POLICY_PROXY_V0` (not the recovered Genesis policy); B2 DAISY challenges are inference-time, not weight training; B0 makes 1 call per item versus 3 for B1–B3.

In [ ]:
science=raw.get("science",[])
if not science: print("NOT_TESTED — no scored Lane B panels")
for fam in sorted({s["backend_family"] for s in science}):
 print("==", fam, "(SYNTHETIC: plumbing only, not model performance)" if fam=="SYNTHETIC_FIXTURE" else "")
 for s in (x for x in science if x["backend_family"]==fam):
  for p in s["panels"]:
   if p["state"]!="SCORED": print("  c=%s REFUSED: %s" % (p["concurrency"], p["reason"]))
  for r in s["rows"]:
   print("  %s c=%s %-18s exact=%s/%s hamming=%.3f abstain=%.3f calls=%s tok/correct=%s src=%s" % (s["served_model_requested"], r["concurrency"], r["arm"], r["correct"], r["n_items"], r["mean_hamming_error"], r["abstention_rate"], r["model_calls"], r["tokens_per_correct_item"], r["source_class"]))
  for c in s["contrasts"]:
   print("  c=%s %s vs %s: A-only=%s B-only=%s p_holm=%.3f" % (c["concurrency"], c["A"], c["B"], c["A_only_correct"], c["B_only_correct"], c["p_holm"]))
print("ANTICUBE NOT_COMPUTED | ΔG* NOT_COMPUTED | PILOT_NONCONFIRMATORY")

## 5. Next actions

Run `tools/concurrency/run_concurrency.py` with `--execute` against a verified authorized endpoint. Preserve `cXX_paired_for_scoring` SHA commitments. Claude independently scores them on the held-out oracle **after** answer commitment, recording exact-match, paired discordance and tokens per correct item. Never infer DAISY weight training from inference-time challenges.